In [22]:
import streamlit as st
import pandas as pd
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb

The full dataset contains over 1.8 million records, which would slow down training and testing greatly to process in full. For the sake of our assignment, we will be only be taking 100000 random rows from the combined dataset.

In [26]:
# Filepaths

train_path = 'fraudTrain.csv'
test_path = 'fraudTest.csv'


# Load the files

df_train = pd.read_csv(train_path, index_col=0)
df_test = pd.read_csv(test_path, index_col=0)


# Combine into single dataframe

df = pd.concat([df_train, df_test], axis=0, ignore_index=True)


# Take a sample of 100000 rows

df = df.sample(100000, random_state=42)

In [27]:
df.head(10)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1541144,2020-09-18 07:13:39,5359543825610251,"fraud_Jenkins, Hauck and Friesen",gas_transport,59.91,Michael,Francis,M,1833 Jeanette Stravenue,Belgrade,MT,59714,45.7801,-111.1439,18182,"Engineer, drilling",1975-06-29,784eb609215cbaf3725642a7a9f8bb57,1379488419,45.274075,-111.649432,0
1731581,2020-12-05 17:48:25,5540636818935089,fraud_Jast-McDermott,shopping_pos,3.96,Kenneth,Foster,M,329 Michael Extension,Lawrence,MA,1843,42.6911,-71.1605,76383,Geoscientist,1985-04-04,74e0ceb1e273a4a37f56dd2b5e6ce03d,1386265705,43.356278,-71.008959,0
354659,2019-06-15 11:24:44,2720894374956739,fraud_Bartoletti-Wunsch,gas_transport,51.17,Audrey,Hickman,F,3325 Gregory Square,Mount Clemens,MI,48043,42.5978,-82.8823,16305,"Psychologist, sport and exercise",1927-05-25,93c2b2ce1e96db464d9a7c7f2e9fb1dd,1339759484,42.372483,-83.508020,0
1493788,2020-08-29 22:50:25,6011438889172900,"fraud_Roob, Conn and Tremblay",shopping_pos,2.06,Allison,Allen,F,40624 Rebecca Spurs,De Witt,AR,72042,34.2853,-91.3336,5161,Electrical engineer,1993-04-08,a269c6b9d9a1603bee720d3ad4fdcfa8,1377816625,33.833389,-91.158293,0
468148,2019-07-25 15:50:35,60495593109,"fraud_Kilback, Nitzsche and Leffler",travel,6.58,Randall,Dillon,M,4440 George Mills Suite 591,Dallas,TX,75210,32.7699,-96.7430,1263321,Television camera operator,1942-11-24,9ce4d94b27e36cc2c79bb02351831722,1343231435,32.458643,-96.577001,0
1389854,2020-07-23 17:29:49,6011477612335392,"fraud_Cormier, Stracke and Thiel",entertainment,24.49,Anthony,Roberts,M,95488 Cabrera Well,Maria Stein,OH,45860,40.4062,-84.5076,2274,"Designer, television/film set",1943-12-17,a9dd902c9f0aabd11730743a7add3bf9,1374600589,39.503889,-84.030238,0
1760906,2020-12-11 23:06:59,2703186189652095,fraud_Skiles LLC,home,99.74,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,46a5dc739d307fa54fa4ffd082e8cad7,1386803219,36.821934,-81.063288,0
469447,2019-07-26 04:56:44,4839615922685395,fraud_Vandervort-Funk,grocery_pos,147.12,Phillip,Robertson,M,85344 Smith Gateway Apt. 280,Harrodsburg,IN,47434,39.0130,-86.5457,76,Social researcher,1955-05-06,cd7e5a8bc6776c1eb4728b01bfe7554f,1343278604,38.202346,-86.136019,0
1835018,2020-12-28 07:36:56,3553629419254918,fraud_Sporer Inc,gas_transport,46.85,Sharon,Johnson,F,7202 Jeffrey Mills,Conway,WA,98238,48.3400,-122.3456,85,"Research officer, political party",1984-09-01,add8330b0172e65570115c36891b71a8,1388216216,47.631938,-122.506513,0
240727,2019-04-30 15:08:10,4481131401752,"fraud_Wintheiser, Dietrich and Schimmel",misc_pos,33.56,Frank,Foster,M,37910 Ward Lights,Shrewsbury,MA,1545,42.2848,-71.7205,35299,English as a second language teacher,1975-04-30,5151c60669d29fb73ec3690489e020f8,1335798490,42.740677,-72.593286,0


In [28]:
df["merchant"] = df["merchant"].apply(lambda x : str(x).replace("fraud_", ""))

In [29]:
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1541144,2020-09-18 07:13:39,5359543825610251,"Jenkins, Hauck and Friesen",gas_transport,59.91,Michael,Francis,M,1833 Jeanette Stravenue,Belgrade,MT,59714,45.7801,-111.1439,18182,"Engineer, drilling",1975-06-29,784eb609215cbaf3725642a7a9f8bb57,1379488419,45.274075,-111.649432,0
1731581,2020-12-05 17:48:25,5540636818935089,Jast-McDermott,shopping_pos,3.96,Kenneth,Foster,M,329 Michael Extension,Lawrence,MA,1843,42.6911,-71.1605,76383,Geoscientist,1985-04-04,74e0ceb1e273a4a37f56dd2b5e6ce03d,1386265705,43.356278,-71.008959,0
354659,2019-06-15 11:24:44,2720894374956739,Bartoletti-Wunsch,gas_transport,51.17,Audrey,Hickman,F,3325 Gregory Square,Mount Clemens,MI,48043,42.5978,-82.8823,16305,"Psychologist, sport and exercise",1927-05-25,93c2b2ce1e96db464d9a7c7f2e9fb1dd,1339759484,42.372483,-83.508020,0
1493788,2020-08-29 22:50:25,6011438889172900,"Roob, Conn and Tremblay",shopping_pos,2.06,Allison,Allen,F,40624 Rebecca Spurs,De Witt,AR,72042,34.2853,-91.3336,5161,Electrical engineer,1993-04-08,a269c6b9d9a1603bee720d3ad4fdcfa8,1377816625,33.833389,-91.158293,0
468148,2019-07-25 15:50:35,60495593109,"Kilback, Nitzsche and Leffler",travel,6.58,Randall,Dillon,M,4440 George Mills Suite 591,Dallas,TX,75210,32.7699,-96.7430,1263321,Television camera operator,1942-11-24,9ce4d94b27e36cc2c79bb02351831722,1343231435,32.458643,-96.577001,0
